## Analysis notebook for the TCA cycle results
loads estimated fluxes, computes summary statistics (median, quantiles), and formats a flux results table.

In [1]:
import pickle
from pathlib import Path

import numpy as np
from jax import config

config.update("jax_enable_x64", True)

import polars as pl

In [2]:
RESULTS_DIR = Path("..") / "data" / "results" / "tca_cycle"
PLOT_DIR = Path("..") / "figures"
true_flux = np.load(RESULTS_DIR / "true_flux.npy", allow_pickle=True).item()
with open(RESULTS_DIR / "estimated_fluxes_full.pkl", "rb") as f:
    estimated_fluxes_full = pickle.load(f)

In [3]:
exclude = {"v1", "v16", "v17", "v18", "v19", "v20"}
flux_keys = [k for k in sorted(true_flux.keys(), key=lambda k: int(k[1:])) if k not in exclude]

rows = []
for model in ("trad", "ilr"):
    arr = np.array(estimated_fluxes_full[model])  # shape (N, n_flux)
    for i, flux_name in enumerate(flux_keys):
        col = arr[:, list(true_flux.keys()).index(flux_name)]
        rows.append({
            "model": model,
            "flux": flux_name,
            "true_value": float(true_flux[flux_name]),
            "median": float(np.median(col)),
            "min": float(np.min(col)),
            "max": float(np.max(col)),
            "q25": float(np.quantile(col, 0.25)),
            "q75": float(np.quantile(col, 0.75)),
        })

summary_df = pl.DataFrame(rows)
summary_df.write_csv(RESULTS_DIR / "estimated_fluxes_full_summary.csv")
print("Summary statistics for estimated fluxes in the TCA cycle:")
print(summary_df.to_pandas().to_string(index=False))

Summary statistics for estimated fluxes in the TCA cycle:


model flux  true_value   median          min      max      q25      q75
 trad   v2       0.201 0.201084 1.702538e-01 0.231667 0.196019 0.206321
 trad   v3       0.621 0.621084 5.902538e-01 0.651667 0.616019 0.626321
 trad   v4       0.621 0.621084 5.902538e-01 0.651667 0.616019 0.626321
 trad   v5       0.621 0.621084 5.902538e-01 0.651667 0.616019 0.626321
 trad   v6       0.592 0.592168 5.305076e-01 0.653334 0.582038 0.602641
 trad   v7       0.807 0.808289 5.497873e-01 1.710388 0.738149 0.897382
 trad   v8       0.215 0.216375 3.942625e-17 1.126790 0.150507 0.300532
 trad   v9       0.296 0.295543 1.753108e-01 0.539047 0.268176 0.325033
 trad  v10       0.475 0.474392 3.592526e-01 0.709785 0.447934 0.502859
 trad  v11       1.213 1.213252 1.120761e+00 1.305000 1.198057 1.228962
 trad  v12       0.296 0.296084 2.652538e-01 0.326667 0.291019 0.301321
 trad  v13       1.509 1.509336 1.386015e+00 1.631667 1.489075 1.530283
 trad  v14       1.735 1.735420 1.581269e+00 1.888334 1.710094 1

In [4]:
# # The following code produces LaTex tables for summary statistics. 
# col_rename = {
#     "flux": "Flux",
#     "true_value": "True value",
#     "median": "Median",
#     "min": "Min",
#     "max": "Max",
#     "q25": "Q25",
#     "q75": "Q75",
# }

# for model, label in (("trad", "Traditional I-MFA"), ("ilr", "Compositional I-MFA")):
#     df = (
#         summary_df.filter(pl.col("model") == model)
#         .drop("model")
#         .rename(col_rename)
#         .to_pandas()
#     )
#     latex = df.to_latex(
#         index=False,
#         float_format="%.3f",
#         caption=f"Estimated fluxes — {label}",
#         label=f"tab:fluxes_{model}",
#     )
#     print(f"% --- {label} ---")
#     print(latex)